<a href="https://colab.research.google.com/github/Zeldano118/QPon_NLP_PBA/blob/main/01_Scraping_Tokenisasi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a
href="https://colab.research.google.com/github/zeldano118/QPon_NLP_PBA/blob/main/notebooks/01_S
craping_Tokenisasi.ipynb" target="_parent"><img
src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 1: Scraping Ulasan QPon + Pengantar NLP & Tokenisasi
## Informasi Aplikasi
- **Nama Aplikasi:** QPon
- **Link Google Play:** [QPon di Play
Store](https://play.google.com/store/apps/details?id=com.qpon.platform)
- **ID Aplikasi:** `com.qpon.platform`
## Identitas
- **Nama:** [Nama Kamu]
- **NRP:** [NRP Kamu]
- **Mata Kuliah:** Pengolahan Bahasa Alami (PBA)
- **Dosen:** Irmasari Hafidz
## Tujuan
1. Mengambil (scraping) seluruh ulasan QPon dari Google Play Store
2. Mempelajari konsep dasar NLP: tokenisasi, korpus, pipeline NLP
3. Menerapkan tokenisasi pada korpus Gutenberg (Inggris) dan IndoNLU SMSA (Indonesia)
4. Menerapkan tokenisasi + demo preprocessing pada ulasan QPon
5. Menyimpan data mentah untuk preprocessing di Week 2
## Deskripsi Proses
1. **Scraping:** `google_play_scraper` → ambil semua ulasan QPon (`lang='id'`)
2. **Tokenisasi Inggris:** Korpus Gutenberg (NLTK) → tokenisasi teks klasik
3. **Tokenisasi Indonesia:** Dataset IndoNLU SMSA (Hugging Face) → tokenisasi teks media
sosial
4. **Tokenisasi QPon:** Menerapkan tokenisasi + stemming + stopword removal pada ulasan QPon
## Referensi
- Jurafsky & Martin, *Speech and Language Processing* (3rd ed., 2023)
- Cahyawijaya et al., *IndoNLU* (ACL 2020)
- Bird et al., *NLP with Python* (NLTK Book, 2009)

In [4]:
# ============================================
# INSTALASI LIBRARY
# ============================================
!pip install google-play-scraper PySastrawi nltk datasets -q

In [5]:
# ============================================
# IMPORT LIBRARY
# ============================================
import pandas as pd
import numpy as np
from google_play_scraper import reviews_all, Sort
from google_play_scraper import app as app_info
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from datasets import load_dataset
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from collections import Counter
import re
import time
import os
# Download resource NLTK
nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
print('Semua library berhasil di-import!')

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


Semua library berhasil di-import!


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


---
## Bagian A: Scraping Ulasan QPon dari Google Play
Mengambil **semua** ulasan aplikasi QPon dari Google Play Store menggunakan
`google_play_scraper`.
- **App ID:** `com.qpon.platform`
- **Bahasa:** Indonesia (`lang='id'`)
- **Metode:** `reviews_all()` untuk mengambil semua ulasan
- **Urutan:** Terbaru (`Sort.NEWEST`)

In [6]:
# ============================================
# INFO APLIKASI QPON
# ============================================
# Mengambil informasi umum aplikasi dari Google Play
qpon_info = app_info('com.qpon.platform', lang='id', country='id')
print('=== INFORMASI APLIKASI QPON ===')
print(f"Nama : {qpon_info['title']}")
print(f"Developer : {qpon_info['developer']}")
print(f"Rating : {qpon_info['score']:.2f}")
print(f"Total Reviews : {qpon_info['reviews']:,}")
print(f"Total Install : {qpon_info['installs']}")
print(f"Genre : {qpon_info['genre']}")
print(f"Version : {qpon_info['version']}")

=== INFORMASI APLIKASI QPON ===
Nama : Qpon-Selalu Ada Diskon
Developer : QPON DIGITAL SINGAPORE
Rating : 3.50
Total Reviews : 5,190
Total Install : 10.000.000+
Genre : Wisata
Version : 2.23.0.0


In [7]:
# ============================================
# SCRAPING SEMUA ULASAN QPON
# ============================================
# reviews_all() mengambil SEMUA ulasan (bukan cuma 200)
# Proses bisa memakan waktu beberapa menit
print('Mulai scraping ulasan QPon...')
print('(Proses ini bisa memakan waktu beberapa menit)')
print()
start_time = time.time()
try:
 qpon_reviews = reviews_all(
 'com.qpon.platform', # App ID QPon
 lang='id', # Bahasa Indonesia
 country='id', # Region Indonesia
 sort=Sort.NEWEST, # Urut terbaru
 sleep_milliseconds=100 # Jeda 100ms (anti-ban)
 )
 elapsed = time.time() - start_time
 print(f'Scraping selesai dalam {elapsed:.1f} detik')
 print(f'Total ulasan berhasil di-scrape: {len(qpon_reviews):,}')
except Exception as e:
 print(f'Error saat scraping: {e}')
 qpon_reviews = []


Mulai scraping ulasan QPon...
(Proses ini bisa memakan waktu beberapa menit)

Scraping selesai dalam 1.8 detik
Total ulasan berhasil di-scrape: 4,637


In [8]:
# ============================================
# KONVERSI KE DATAFRAME & SIMPAN CSV
# ============================================
df_qpon = pd.DataFrame(qpon_reviews)
print('=== INFO DATASET QPON ===')
print(f'Jumlah ulasan : {df_qpon.shape[0]:,}')
print(f'Jumlah kolom : {df_qpon.shape[1]}')
print(f'Kolom : {df_qpon.columns.tolist()}')
print()
# Info tipe data
print('=== TIPE DATA ===')
print(df_qpon.dtypes)
print()
# Simpan ke CSV
df_qpon.to_csv('qpon_reviews_raw.csv', index=False, encoding='utf-8')
print(f'Data disimpan ke qpon_reviews_raw.csv')
print(f'Ukuran file: {os.path.getsize("qpon_reviews_raw.csv") / 1024:.1f} KB')

=== INFO DATASET QPON ===
Jumlah ulasan : 4,637
Jumlah kolom : 11
Kolom : ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

=== TIPE DATA ===
reviewId                        object
userName                        object
userImage                       object
content                         object
score                            int64
thumbsUpCount                    int64
reviewCreatedVersion            object
at                      datetime64[ns]
replyContent                    object
repliedAt               datetime64[ns]
appVersion                      object
dtype: object

Data disimpan ke qpon_reviews_raw.csv
Ukuran file: 2129.0 KB


In [9]:
# ============================================
# EKSPLORASI AWAL DATA QPON
# ============================================
# 5 ulasan pertama
print('=== 5 ULASAN PERTAMA ===')
for i, row in df_qpon.head().iterrows():
 print(f"\n--- Ulasan {i+1} ---")
 print(f"User : {row.get('userName', 'N/A')}")
 print(f"Rating : {'\u2B50' * row.get('score', 0)} ({row.get('score', 'N/A')}/5)")
 print(f"Tanggal: {row.get('at', 'N/A')}")
 print(f"Ulasan : {row.get('content', 'N/A')}")
print()
print('=== DISTRIBUSI RATING ===')
rating_dist = df_qpon['score'].value_counts().sort_index()
for rating, count in rating_dist.items():
 pct = count / len(df_qpon) * 100
 bar = '\u2588' * int(pct / 2)
 print(f" {'\u2B50'*rating} ({rating}): {count:>6,} ulasan ({pct:5.1f}%) {bar}")
print()
print(f'Rating rata-rata: {df_qpon["score"].mean():.2f}')

=== 5 ULASAN PERTAMA ===

--- Ulasan 1 ---
User : Pengguna Google
Rating : ⭐ (1/5)
Tanggal: 2026-03-02 05:53:45
Ulasan : apk fomo doang bisa sekarang susah banyak promo tapi gak bisa di pake buang buang duit doang.

--- Ulasan 2 ---
User : Pengguna Google
Rating : ⭐⭐⭐⭐⭐ (5/5)
Tanggal: 2026-03-02 05:02:22
Ulasan : Harga terjangkau Dengan rasa dan kualitas sungguh memukau.. i like momoyo. Coba dulu baru komen.. dah Di jamin manyosss Dan ketagihan

--- Ulasan 3 ---
User : Pengguna Google
Rating : ⭐ (1/5)
Tanggal: 2026-03-01 08:54:47
Ulasan : kok pas mau pesen sering logout sendiri yg akun Baru

--- Ulasan 4 ---
User : Pengguna Google
Rating : ⭐⭐⭐⭐⭐ (5/5)
Tanggal: 2026-03-01 05:33:12
Ulasan : Qpon ngebantu banget buat jajan, 1 waktu aku punya voucher yg lupa beli di pakai, dan ternyata sudah lewat masa expired nya, panik dong, chat cs qpon, Alhamdulillah bisa di bantu refund, makasih qpon 💚🙏

--- Ulasan 5 ---
User : Pengguna Google
Rating : ⭐⭐⭐⭐⭐ (5/5)
Tanggal: 2026-03-01 04:35:17
Ulasan :

In [11]:
# ============================================
# SAMPEL ULASAN PER RATING
# ============================================
for rating in range(1, 6):
    subset = df_qpon[df_qpon['score'] == rating]
    if len(subset) > 0:
        sample = subset['content'].dropna().head(2).tolist()
        print(f"\n{'='*60}")
        print(f"RATING {'\u2B50'*rating} ({rating}/5) - {len(subset):,} ulasan")
        print(f"{'='*60}")
        for j, text in enumerate(sample):
            display_text = str(text)[:200] + '...' if len(str(text)) > 200 else str(text)
            print(f" Contoh {j+1}: {display_text}")


RATING ⭐ (1/5) - 2,031 ulasan
 Contoh 1: apk fomo doang bisa sekarang susah banyak promo tapi gak bisa di pake buang buang duit doang.
 Contoh 2: kok pas mau pesen sering logout sendiri yg akun Baru

RATING ⭐⭐ (2/5) - 267 ulasan
 Contoh 1: duit gua kgk blik anjir salah beli juga minta tanggung jawab malah gk jelas balesan nya
 Contoh 2: tolong buat menu riwayat transaksinya dicantumkan nama outlet juga di bagian depannya , sehingga ga perlu dibuka dulu satu2 dan klik detail pesanan.

RATING ⭐⭐⭐ (3/5) - 169 ulasan
 Contoh 1: karena pas flash sale yang di liat hanya brand aja alhasil gak bisa kepake dan jeleknya GAK ADA OPTION REFUND . saya di bandung salah beli voucher malah yang jakarta .. tolong dong qpon kasih kebijaka...
 Contoh 2: kenapa sekarang update terbaru malah tenant² nya gak sesuai lokasi kita ya?, biasanya merunut dr yg terdekat, ini langsung merchant ibu kota aja. Dan gak ada lagi sort berdasarkan jarak, why ?

RATING ⭐⭐⭐⭐ (4/5) - 130 ulasan
 Contoh 1: saeee
 Contoh 2: m

---
## Bagian B: Tokenisasi Teks Inggris (Korpus Gutenberg)
Menggunakan korpus bawaan NLTK (Gutenberg corpus) yang berisi buku klasik.
Sesuai materi Week 1: *Introduction to NLP – Tokenization* (Irmasari Hafidz).
**Referensi:** Bird et al., *NLP with Python* (NLTK Book, 2009), Chapter 1.

In [12]:
# ============================================
# TOKENISASI TEKS INGGRIS (GUTENBERG)
# ============================================
# Sesuai materi dosen: menggunakan korpus Gutenberg
# Ref: Bird et al., NLP with Python (2009)
text_en = nltk.corpus.gutenberg.raw('austen-emma.txt')[:1000]
print('=== TEKS ASLI (BEFORE) ===')
print(text_en[:500])
print()
# Tokenisasi
tokens_en = word_tokenize(text_en)
print('=== HASIL TOKENISASI (AFTER) ===')
print('20 token pertama:', tokens_en[:20])
print()
print('=== STATISTIK ===')
print(f'Total token : {len(tokens_en)}')
print(f'Token unik : {len(set(tokens_en))}')
print(f'Panjang teks : {len(text_en)} karakter')

=== TEKS ASLI (BEFORE) ===
[Emma by Jane Austen 1816]

VOLUME I

CHAPTER I


Emma Woodhouse, handsome, clever, and rich, with a comfortable home
and happy disposition, seemed to unite some of the best blessings
of existence; and had lived nearly twenty-one years in the world
with very little to distress or vex her.

She was the youngest of the two daughters of a most affectionate,
indulgent father; and had, in consequence of her sister's marriage,
been mistress of his house from a very early period.  Her mother
had died t

=== HASIL TOKENISASI (AFTER) ===
20 token pertama: ['[', 'Emma', 'by', 'Jane', 'Austen', '1816', ']', 'VOLUME', 'I', 'CHAPTER', 'I', 'Emma', 'Woodhouse', ',', 'handsome', ',', 'clever', ',', 'and', 'rich']

=== STATISTIK ===
Total token : 198
Token unik : 114
Panjang teks : 1000 karakter


---
## Bagian C: Tokenisasi Teks Indonesia (IndoNLU SMSA)
Menggunakan dataset **IndoNLU SMSA** (Social Media Sentiment Analysis) dari Hugging Face.
Dataset ini berisi teks media sosial Indonesia dengan label sentimen.
**Referensi:** Cahyawijaya et al., *IndoNLU: Benchmark and Resources for Evaluating
Indonesian NLP* (ACL 2020).

In [14]:
# ============================================
# TOKENISASI TEKS INDONESIA (INDONLU SMSA)
# ============================================
# Dataset: kornwtp/indonlu-smsa dari Hugging Face
# Berisi teks media sosial Indonesia + label sentimen
dataset = load_dataset('kornwtp/indonlu-smsa', split='train')
df_indo = dataset.to_pandas()
print(f'Jumlah data IndoNLU SMSA: {len(df_indo)}')
print(f'Kolom: {list(df_indo.columns)}')
print()
# Ambil 3 sampel teks + label
text_col = 'text' if 'text' in df_indo.columns else df_indo.columns[0]
label_col = 'label' if 'label' in df_indo.columns else df_indo.columns[1]
sample_texts_indo = df_indo[text_col][:3].tolist()
sample_labels = df_indo[label_col][:3].tolist()
# Inisialisasi stemmer Sastrawi
stemmer = StemmerFactory().create_stemmer()
# Tokenisasi + stemming untuk setiap sampel
for i, text in enumerate(sample_texts_indo):
 print(f'\nContoh Teks {i+1} (Label: {sample_labels[i]}):')
 print(f'Teks Asli: {text}')

 # Tokenisasi
 tokens = word_tokenize(str(text))
 print(f'Token: {tokens}')
 print(f'Jumlah Token: {len(tokens)}')
 print(f'Token Unik: {len(set(tokens))}')

 # Stemming
 stemmed_text = stemmer.stem(str(text))
 stemmed_tokens = word_tokenize(stemmed_text)
 print(f'Token setelah Stemming: {stemmed_tokens}')
 print(f'Jumlah Token setelah Stemming: {len(stemmed_tokens)}')

 # Simpan token ke file
 with open(f'tokens_sample_{i+1}.txt', 'w', encoding='utf-8') as f:
  f.write('\n'.join(tokens))
  print(f'Token disimpan ke tokens_sample_{i+1}.txt')
# Analisis frekuensi kata
all_tokens_indo = []
for text in sample_texts_indo:
 all_tokens_indo.extend(word_tokenize(str(text).lower()))
word_freq_indo = Counter(all_tokens_indo).most_common(10)
print(f'\n10 Kata Teratas: {word_freq_indo}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/687 [00:00<?, ?B/s]

data/train-00000-of-00001-026a27ce8c4508(…):   0%|          | 0.00/1.28M [00:00<?, ?B/s]

data/validation-00000-of-00001-495bab7e4(…):   0%|          | 0.00/148k [00:00<?, ?B/s]

data/test-00000-of-00001-8e9d52f9ccaaffe(…):   0%|          | 0.00/45.1k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1260 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Jumlah data IndoNLU SMSA: 11000
Kolom: ['texts', 'labels']


Contoh Teks 1 (Label: 0):
Teks Asli: warung ini dimiliki oleh pengusaha pabrik tahu yang sudah puluhan tahun terkenal membuat tahu putih di bandung . tahu berkualitas , dipadu keahlian memasak , dipadu kretivitas , jadilah warung yang menyajikan menu utama berbahan tahu , ditambah menu umum lain seperti ayam . semuanya selera indonesia . harga cukup terjangkau . jangan lewatkan tahu bletoka nya , tidak kalah dengan yang asli dari tegal !
Token: ['warung', 'ini', 'dimiliki', 'oleh', 'pengusaha', 'pabrik', 'tahu', 'yang', 'sudah', 'puluhan', 'tahun', 'terkenal', 'membuat', 'tahu', 'putih', 'di', 'bandung', '.', 'tahu', 'berkualitas', ',', 'dipadu', 'keahlian', 'memasak', ',', 'dipadu', 'kretivitas', ',', 'jadilah', 'warung', 'yang', 'menyajikan', 'menu', 'utama', 'berbahan', 'tahu', ',', 'ditambah', 'menu', 'umum', 'lain', 'seperti', 'ayam', '.', 'semuanya', 'selera', 'indonesia', '.', 'harga', 'cukup', 'terjangkau', '.', 'jang

---
## Bagian D: Tokenisasi pada Ulasan QPon
Menerapkan tokenisasi dan demo preprocessing pada data ulasan QPon yang sudah di-scrape di
Bagian A.
Ini menghubungkan konsep NLP dari materi dosen dengan data proyek kita.

In [16]:
# ============================================
# TOKENISASI ULASAN QPON
# ============================================
sample_reviews = df_qpon['content'].dropna().head(5).tolist()
print('=== TOKENISASI ULASAN QPON (BEFORE vs AFTER) ===')
print()
for i, review in enumerate(sample_reviews):
 print(f'--- Ulasan QPon #{i+1} ---')
 print(f'BEFORE : {review}')
 tokens = word_tokenize(str(review))
 print(f'AFTER : {tokens}')
 print(f'Jumlah token: {len(tokens)}')
 print()
# Frekuensi token seluruh ulasan
all_tokens_qpon = []
for review in df_qpon['content'].dropna():
 all_tokens_qpon.extend(word_tokenize(str(review).lower()))
print(f'Total token dari semua ulasan QPon: {len(all_tokens_qpon):,}')
print(f'Token unik: {len(set(all_tokens_qpon)):,}')
print()
freq_qpon = Counter(all_tokens_qpon)
print('=== 15 TOKEN PALING SERING DI ULASAN QPON ===')
for word, count in freq_qpon.most_common(15):
 print(f' {word:20s} : {count:>6,} kali')

=== TOKENISASI ULASAN QPON (BEFORE vs AFTER) ===

--- Ulasan QPon #1 ---
BEFORE : apk fomo doang bisa sekarang susah banyak promo tapi gak bisa di pake buang buang duit doang.
AFTER : ['apk', 'fomo', 'doang', 'bisa', 'sekarang', 'susah', 'banyak', 'promo', 'tapi', 'gak', 'bisa', 'di', 'pake', 'buang', 'buang', 'duit', 'doang', '.']
Jumlah token: 18

--- Ulasan QPon #2 ---
BEFORE : Harga terjangkau Dengan rasa dan kualitas sungguh memukau.. i like momoyo. Coba dulu baru komen.. dah Di jamin manyosss Dan ketagihan
AFTER : ['Harga', 'terjangkau', 'Dengan', 'rasa', 'dan', 'kualitas', 'sungguh', 'memukau', '..', 'i', 'like', 'momoyo', '.', 'Coba', 'dulu', 'baru', 'komen', '..', 'dah', 'Di', 'jamin', 'manyosss', 'Dan', 'ketagihan']
Jumlah token: 24

--- Ulasan QPon #3 ---
BEFORE : kok pas mau pesen sering logout sendiri yg akun Baru
AFTER : ['kok', 'pas', 'mau', 'pesen', 'sering', 'logout', 'sendiri', 'yg', 'akun', 'Baru']
Jumlah token: 10

--- Ulasan QPon #4 ---
BEFORE : Qpon ngebantu bange

In [17]:
# ============================================
# DEMO STEMMING & STOPWORD REMOVAL PADA QPON
# ============================================
# --- Stemming ---
long_words = [w for w in set(all_tokens_qpon) if len(w) > 6 and w.isalpha()]
sample_words = sorted(long_words, key=lambda w: freq_qpon[w], reverse=True)[:12]
print('=== DEMO STEMMING KATA-KATA DARI ULASAN QPON ===')
print(f'{"Kata Asli":25s} {"Frek":>6s} -> Hasil Stem')
print('-' * 55)
for kata in sample_words:
 stem = stemmer.stem(kata)
 changed = ' *' if stem != kata else ''
 print(f'{kata:25s} {freq_qpon[kata]:>6,} -> {stem}{changed}')
print()
# --- Stopword Removal ---
stop_id = set(stopwords.words('indonesian'))
print(f'Jumlah stopwords Indonesia di NLTK: {len(stop_id)}')
print()
print('=== STOPWORD REMOVAL PADA 3 ULASAN QPON ===')
for i in range(min(3, len(sample_reviews))):
 review = str(sample_reviews[i])
 tokens = word_tokenize(review.lower())
 filtered = [w for w in tokens if w not in stop_id and w.isalpha()]
 print(f'\n--- Ulasan QPon #{i+1} ---')
 print(f'BEFORE ({len(tokens)} token): {tokens}')
 print(f'AFTER ({len(filtered)} token): {filtered}')
 print(f'Dihapus: {len(tokens) - len(filtered)} stopwords/tanda baca')

=== DEMO STEMMING KATA-KATA DARI ULASAN QPON ===
Kata Asli                   Frek -> Hasil Stem
-------------------------------------------------------
aplikasi                   1,408 -> aplikasi
voucher                      794 -> voucher
makanan                      320 -> makan *
padahal                      291 -> padahal
membantu                     218 -> bantu *
gampang                      217 -> gampang
download                     184 -> download
dipakai                      183 -> pakai *
aplikasinya                  167 -> aplikasi *
restoran                     143 -> restoran
langsung                     125 -> langsung
terlalu                      117 -> terlalu

Jumlah stopwords Indonesia di NLTK: 757

=== STOPWORD REMOVAL PADA 3 ULASAN QPON ===

--- Ulasan QPon #1 ---
BEFORE (18 token): ['apk', 'fomo', 'doang', 'bisa', 'sekarang', 'susah', 'banyak', 'promo', 'tapi', 'gak', 'bisa', 'di', 'pake', 'buang', 'buang', 'duit', 'doang', '.']
AFTER (11 token): ['apk', 'fomo', 

---
## Kesimpulan Week 1
### Data yang Dikumpulkan
- Berhasil scraping **[isi jumlah]** ulasan QPon dari Google Play Store
- App ID: `com.qpon.platform`, Bahasa: Indonesia
- Data disimpan di: `qpon_reviews_raw.csv`
### Konsep NLP yang Dipelajari
1. **NLP** = bidang AI untuk memproses bahasa manusia (Jurafsky & Martin, 2023)
2. **Tokenisasi** = memecah teks jadi token, langkah pertama NLP (Bird et al., 2009)
3. **Korpus** = koleksi teks terstruktur: Gutenberg (EN), IndoNLU SMSA (ID) (Cahyawijaya et
al., 2020)
4. **Stemming** = mengembalikan kata berimbuhan ke bentuk dasar (Sastrawi)
5. **Stopwords** = kata umum yang dihapus agar analisis fokus ke kata bermakna
6. **Pipeline NLP** = Data > Preprocessing > Feature Extraction > Modeling > Evaluation
### Tantangan NLP Bahasa Indonesia
- Slang dan bahasa informal
- Afiksasi kompleks (me-, ber-, di-, ter-, pe-, -kan, -an, -i)
- Code-switching (campur bahasa Inggris)
- Dataset beranotasi masih terbatas (low-resource language)
### Preview Week 2
- Preprocessing lengkap pada seluruh ulasan QPon
- Sentiment labeling (1-2=Negatif, 3=Netral, 4-5=Positif)
- EDA + visualisasi (word cloud, distribusi, tren)
---
*Notebook ini dibuat sebagai Tugas Week 1 mata kuliah PBA, Sistem Informasi ITS 2026*